<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L53_Cost_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L53: Cost Optimization - Inference Cost and Resource Management

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 53 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Estimate cost per request (tokens, GPU time)
- Apply batching and caching to reduce cost
- Choose model size and quantization for cost-performance trade-off

---

## Concept: Cost Optimization

**Cost** = GPU/time (fixed) + per-token compute. **Reduce**: (1) smaller or quantized models; (2) batching to increase throughput; (3) KV-cache reuse for same prefix; (4) prompt caching; (5) tiered models (cheap for easy, expensive for hard). We measure tokens and time and compute a simple cost metric.

---

In [ ]:
!pip install transformers torch -q
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Measure Tokens and Latency

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt")
start = time.perf_counter()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id)
elapsed = time.perf_counter() - start
n_input = inputs["input_ids"].size(1)
n_output = out.size(1) - n_input
print(f"Input tokens: {n_input}, Output tokens: {n_output}, Time: {elapsed:.2f}s")
print(f"Throughput: {n_output/elapsed:.1f} tokens/s")

## Part 2: Simple Cost Model

---

In [ ]:
# Example: cost = C_fixed per request + C_per_token * total_tokens (input + output)
C_fixed = 0.001
C_per_token = 0.00001
total_tokens = n_input + n_output
cost = C_fixed + C_per_token * total_tokens
print(f"Example cost per request: ${cost:.4f}")
print("Tune C_fixed/C_per_token from your GPU pricing and utilization.")

## Part 3: Strategies

---

In [ ]:
print("Strategies: (1) Batch requests to amortize fixed cost (2) Cache common prefixes (3) Use smaller/quantized model (4) Limit max_new_tokens")

## Exercises

1. Run 10 requests single-threaded; report average latency and tokens/s.
2. Batch 5 prompts and compare total time vs 5x single-request time.
3. Define a cost budget per user per day and derive max requests or max tokens.

---

## Key Takeaways

1. Cost = fixed + per-token; measure both for your deployment.
2. Batching and caching reduce effective cost per request.
3. Model size and quantization directly affect cost and quality.

---

## Next Lesson

**L54: Evaluation Frameworks** – Benchmarks and automated evaluation.

---